# Atividade guiada: Recomendador por Embeddings (MovieLens)

Este notebook é o **roteiro guiado** da atividade "Recomendador por Embeddings", continuação da aula
"Tokens e Embeddings". Ele não contém a solução pronta: cada célula de texto descreve **o que você
precisa fazer** e **por que**, sempre relacionando com um conceito visto em aula (One-Hot, TF-IDF,
embeddings, Word2Vec/skip-gram, similaridade de cosseno). As células de código logo abaixo trazem uma
estrutura inicial com marcações `# COMPLETE AQUI`, que você deve preencher.

A atividade está dividida em seis partes, na mesma ordem do enunciado:

1. Preparação dos dados (tokens)
2. Baseline ortogonal (One-Hot / TF-IDF)
3. Treinamento do Word2Vec
4. Exploração do espaço vetorial (vizinhos, analogia, visualização)
5. Função de recomendação
6. Relatório crítico

**Dataset**: MovieLens `ml-latest-small` (aproximadamente 100 mil avaliações, 9.742 filmes, 610
usuários), disponibilizado gratuitamente pelo GroupLens Research (Universidade de Minnesota), sem
necessidade de login ou cartão de crédito.

Fonte do dataset: F. Maxwell Harper and Joseph A. Konstan. 2015. *The MovieLens Datasets: History and
Context*. ACM Transactions on Interactive Intelligent Systems. Disponível em:
https://grouplens.org/datasets/movielens/latest/

**Importante**: escolha um filme semente próprio (diferente dos colegas) para as Partes 4, 5 e 6, como
pedido no enunciado. O gabarito usa "Jurassic Park (1993)" apenas como exemplo ilustrativo; o seu
resultado nessas partes será diferente disso, e é esperado que seja.

Depois de terminar, você receberá o gabarito para comparar sua solução, principalmente a lógica e as
justificativas escritas, já que os números exatos podem variar entre execuções (o treinamento do
Word2Vec envolve aleatoriedade).


## Passo 0: Preparação do ambiente

Instale as bibliotecas que serão usadas ao longo do notebook: `gensim` (para treinar o Word2Vec),
`scikit-learn` (para TF-IDF, similaridade de cosseno e PCA) e `pandas`/`matplotlib` (para manipulação de
dados e visualização). É o mesmo papel da célula `!pip install gensim` do notebook da aula, só que aqui
já deixamos listadas todas as dependências que você vai precisar.


In [ ]:
#!pip install -q COLOCAR-BIBLIOTECAS


## Parte 1: Preparação dos dados (tokens)

### Passo 1.1: Download e extração do dataset

Acesse https://grouplens.org/datasets/movielens/latest/ e identifique o link direto de download do
arquivo `ml-latest-small.zip` (o link termina em `.../ml-latest-small.zip`). Na célula abaixo:

1. Baixe o arquivo `.zip` para o ambiente do Colab.
2. Extraia o conteúdo do `.zip` em uma pasta local.
3. Confirme, listando os arquivos extraídos, que `ratings.csv` e `movies.csv` estão presentes.

**Dica**: use `urllib.request.urlretrieve(url, caminho_destino)` para baixar o arquivo e
`zipfile.ZipFile(caminho, "r")` com o método `.extractall(...)` para descompactar.


In [ ]:
import urllib.request
import zipfile
import os

url = "___"        # COMPLETE AQUI: URL direto do arquivo ml-latest-small.zip
zip_path = "___"   # COMPLETE AQUI: nome do arquivo local a ser salvo, ex. "ml-latest-small.zip"

# COMPLETE AQUI: baixe o arquivo com urllib.request.urlretrieve

# COMPLETE AQUI: extraia o conteudo do zip com zipfile.ZipFile(...).extractall(...)

# COMPLETE AQUI: liste os arquivos extraidos com os.listdir(...) para conferir


### Passo 1.2: Carregamento em DataFrames

Carregue `ratings.csv` e `movies.csv` com `pandas.read_csv`. Em seguida, defina `movieId` como índice
do DataFrame `movies`, para permitir consultas rápidas de título/gênero a partir do ID (o mesmo papel que
`songs_df.set_index('id')` cumpria no notebook da aula).

Imprima o `shape` dos dois DataFrames e visualize as primeiras linhas de `ratings` para conferir se o
carregamento funcionou.


In [ ]:
import pandas as pd

ratings = ___   # COMPLETE AQUI: carregue "ml-latest-small/ratings.csv" com pd.read_csv
movies = ___    # COMPLETE AQUI: carregue "ml-latest-small/movies.csv" com pd.read_csv
movies = ___    # COMPLETE AQUI: defina "movieId" como indice de movies (metodo set_index)

# COMPLETE AQUI: imprima ratings.shape e movies.shape

# COMPLETE AQUI: exiba as primeiras linhas de ratings (metodo head)


### Passo 1.3: Construção das sequências por usuário

Este é o passo conceitualmente mais importante da Parte 1. Assim como uma **playlist** era uma
sequência de IDs de música no notebook da aula, aqui o **histórico de um usuário** deve virar uma
sequência de IDs de filme, ou seja, a "frase" de entrada que o Word2Vec vai usar para aprender.

Siga estes sub-passos:

1. Filtre `ratings` mantendo apenas avaliações com nota maior ou igual a um limiar que você escolher
   (o gabarito usa 4.0, em uma escala de 0.5 a 5). Justifique essa escolha depois, no relatório da
   Parte 6.
2. Agrupe as avaliações filtradas por `userId` e, para cada usuário, construa a lista de `movieId`
   avaliados positivamente, **convertendo cada ID para `string`** (o `gensim.Word2Vec` espera tokens
   como strings, não números).
3. O resultado final deve ser uma lista de listas: `sequencias`, uma sublista por usuário.

**Dica**: use `groupby("userId")["movieId"].apply(...)` seguido de `.tolist()`.


In [ ]:
NOTA_MINIMA = ___   # COMPLETE AQUI: escolha e justifique um limiar de nota (ex. 4.0)

positivas = ___      # COMPLETE AQUI: filtre ratings mantendo apenas rating >= NOTA_MINIMA

sequencias = ___      # COMPLETE AQUI: agrupe por userId e construa a lista de listas de movieId (como string)

# COMPLETE AQUI: imprima o numero de sequencias e um exemplo (ex. a primeira sequencia)


### Passo 1.4: Estatísticas descritivas pedidas no enunciado

Calcule e reporte:

- Número de usuários (sequências).
- Tamanho médio das sequências.
- Tamanho mínimo e máximo das sequências.
- Número de filmes distintos que aparecem nas avaliações positivas (o "tamanho do vocabulário",
  equivalente ao conceito discutido no slide "Propriedades do Tokenizer": é o universo de tokens que o
  Word2Vec terá que aprender a representar).


In [ ]:
import numpy as np

tamanhos = ___            # COMPLETE AQUI: lista com o tamanho (len) de cada sequencia
filmes_distintos = ___    # COMPLETE AQUI: numero de movieId distintos em "positivas" (metodo nunique)

# COMPLETE AQUI: imprima numero de usuarios, tamanho medio, minimo, maximo e filmes distintos


## Parte 2: Baseline ortogonal (para reforçar por que embeddings importam)

Antes de treinar o Word2Vec, reproduza o problema discutido nos slides "A primeira tentativa One Hot
Encoding" e "O problema da ortogonalidade": construa uma representação **TF-IDF** dos filmes a partir
apenas da coluna `genres`, tratando cada gênero (separados por `|`) como um "termo" do documento
"filme". Essa representação é esparsa e baseada em coincidência exata de palavras, sem nenhuma noção
aprendida de similaridade semântica.

Passos:

1. Use `TfidfVectorizer` com um `token_pattern` que separe corretamente os gêneros por `|` (não pelo
   espaço em branco padrão).
2. Ajuste (`fit_transform`) o vetorizador sobre a coluna `genres` de `movies`.
3. Escreva uma função `similaridade_tfidf(id_a, id_b)` que retorne a similaridade de cosseno entre os
   vetores TF-IDF de dois filmes.
4. Escolha dois filmes que você **sabe** serem parecidos em conteúdo/público, mas que **não
   compartilham exatamente os mesmos gêneros** na base (por exemplo, dois filmes de ação de décadas
   diferentes, ou um filme de terror clássico e um mais recente). Calcule e imprima a similaridade
   TF-IDF entre eles.

**Dica**: para buscar o ID de um filme pelo título, use
`movies[movies["title"].str.contains("trecho do titulo")].index[0]`.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = ___          # COMPLETE AQUI: TfidfVectorizer com token_pattern adequado para separar por "|"
matriz_tfidf = ___   # COMPLETE AQUI: fit_transform sobre movies["genres"]

movie_ids = ___       # COMPLETE AQUI: lista de IDs de movies (movies.index.tolist())
posicao = ___          # COMPLETE AQUI: dicionario {movie_id: posicao na matriz}

def similaridade_tfidf(id_a, id_b):
    # COMPLETE AQUI: retorne a similaridade de cosseno entre os vetores TF-IDF de id_a e id_b
    pass

# COMPLETE AQUI: escolha dois filmes (busque os IDs pelo titulo) e imprima seus generos e a similaridade


**Para o seu relatório (Parte 6)**: anote o valor de similaridade obtido e reflita: ele condiz com o
quanto você, subjetivamente, acha esses dois filmes parecidos? Relacione sua resposta com a limitação
apontada no slide "A Falha Semântica (O Limite)".


## Parte 3: Treinamento do Word2Vec

Treine um modelo `Word2Vec` do `gensim` sobre a lista `sequencias` construída na Parte 1. Cada filme
será tratado como um token; cada histórico de usuário, como uma frase, exatamente como no exemplo de
playlists de música da aula.

Antes de escrever o código, decida e **justifique por escrito** (célula de texto, para o relatório da
Parte 6) os valores dos seguintes hiperparâmetros:

- `vector_size`: dimensão do embedding. Pense no equilíbrio entre capacidade de representação e tempo
  de treino em CPU.
- `window`: quantos filmes vizinhos dentro do histórico de um usuário contam como contexto positivo.
- `min_count`: filmes com menos avaliações positivas que esse valor serão ignorados. Por quê isso é
  necessário (relacione com o conceito de tokens raros)?
- `negative`: número de amostras negativas por exemplo positivo (relacione com o slide "Word2Vec" e a
  ideia de *negative sampling*, que evita que o modelo aprenda a prever sempre 1).
- `epochs`: número de passagens pelos dados.

Depois de treinar, imprima o tamanho do vocabulário aprendido (`len(modelo_w2v.wv.key_to_index)`) e
compare com o número de filmes distintos calculado na Parte 1.4. Se os números forem diferentes,
explique por quê (dica: pense no papel do `min_count`).


In [ ]:
from gensim.models import Word2Vec

modelo_w2v = Word2Vec(
    sequencias,
    vector_size=___,   # COMPLETE AQUI
    window=___,        # COMPLETE AQUI
    min_count=___,     # COMPLETE AQUI
    negative=___,      # COMPLETE AQUI
    workers=4,
    seed=42,
    epochs=___,        # COMPLETE AQUI
)

# COMPLETE AQUI: imprima o tamanho do vocabulario aprendido pelo modelo


## Parte 4: Exploração do espaço vetorial

### Passo 4.1: Filme semente e vizinhos mais próximos

Escolha o **seu** filme semente (diferente dos colegas). Escreva uma função auxiliar
`buscar_id_por_titulo(trecho_titulo)` que receba um trecho do título e devolva o `movieId`
correspondente (use `str.contains`, com `case=False` para ignorar maiúsculas/minúsculas).

Em seguida, busque os cinco filmes mais próximos do seu filme semente no espaço de embeddings, usando
`modelo_w2v.wv.most_similar`. Para cada resultado, imprima o escore de similaridade, o título e o gênero
do filme (consulte `movies.loc[...]`).

Isso reproduz o procedimento `model.wv.most_similar(positive=str(song_id))` usado para músicas no
notebook da aula, agora aplicado a filmes.


In [ ]:
def buscar_id_por_titulo(trecho_titulo):
    # COMPLETE AQUI: filtre movies pelo titulo (case=False) e retorne o primeiro movieId encontrado
    # dica: lance um erro (raise ValueError) se nao encontrar nenhum resultado
    pass

filme_semente_id = buscar_id_por_titulo("___")   # COMPLETE AQUI: escolha o SEU filme semente
# COMPLETE AQUI: imprima titulo e genero do filme semente

vizinhos = ___   # COMPLETE AQUI: modelo_w2v.wv.most_similar(...) com topn=5

# COMPLETE AQUI: para cada (movie_id_str, score) em vizinhos, imprima score, titulo e genero


### Passo 4.2: Tentativa de analogia vetorial

Reproduza a ideia do slide "Analogias" (`king - man + woman ~= queen`), agora tentando uma analogia de
gênero cinematográfico com filmes da sua escolha. Lembre-se: `positive=[A, B], negative=[C]` calcula o
vetor `A - C + B` e busca os filmes mais próximos desse ponto resultante.

Escolha três filmes de forma que a subtração/soma faça sentido conceitualmente para você (por exemplo,
"filme de animação infantil" menos "comédia adulta" mais "terror", tentando chegar em algo do gênero
terror). O resultado pode não ser tão limpo quanto o exemplo clássico de linguagem natural, isso é
esperado: parte da atividade é você analisar criticamente esse resultado no relatório da Parte 6, e não
apenas confirmar que "funcionou".


In [ ]:
filme_a_id = buscar_id_por_titulo("___")   # COMPLETE AQUI
filme_b_id = buscar_id_por_titulo("___")   # COMPLETE AQUI
filme_c_id = buscar_id_por_titulo("___")   # COMPLETE AQUI

# COMPLETE AQUI: use modelo_w2v.wv.most_similar com positive=[a, b] e negative=[c] (topn=5)
resultado_analogia = ___

# COMPLETE AQUI: imprima score, titulo e genero de cada resultado


### Passo 4.3: Visualização do espaço de embeddings em 2D

Reduza os embeddings do seu filme semente e dos vizinhos encontrados no Passo 4.1 de `vector_size`
dimensões para 2 dimensões, usando `PCA(n_components=2)`. Gere um gráfico de dispersão destacando o
filme semente com uma cor diferente dos vizinhos, com o título de cada filme como rótulo.

Lembre-se: a similaridade de cosseno usada na busca é calculada no espaço vetorial original (com todas
as dimensões); a projeção em 2D serve **apenas** para visualização.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

ids_para_plotar = ___   # COMPLETE AQUI: lista com o ID do filme semente + os IDs dos vizinhos
vetores = ___             # COMPLETE AQUI: array numpy com os vetores desses filmes (modelo_w2v.wv[...])

pca = ___     # COMPLETE AQUI: PCA(n_components=2, random_state=42)
coords = ___  # COMPLETE AQUI: fit_transform sobre "vetores"

# COMPLETE AQUI: plote um scatter com coords, destacando o filme semente e anotando os titulos


## Parte 5: Função de recomendação

Encapsule o fluxo do Passo 4.1 em uma função reutilizável `recomendar(movie_id, topn=5)` que retorne um
DataFrame com título, gênero e o escore de similaridade dos `topn` filmes mais próximos do `movie_id`
informado, no mesmo espírito da função `print_recommendations` usada para músicas no notebook da aula.

Teste a função com o seu filme semente.


In [ ]:
def recomendar(movie_id, topn=5):
    # COMPLETE AQUI: busque os "topn" mais similares com modelo_w2v.wv.most_similar
    # COMPLETE AQUI: monte um DataFrame com colunas title, genres e similaridade
    pass

recomendar(filme_semente_id, topn=5)


## Parte 6: Relatório crítico

Escreva, nesta própria célula (edite o texto abaixo), suas respostas para as três perguntas do
enunciado. Use como base os resultados que você obteve nas partes anteriores.

**1. Por que a mesma técnica usada para palavras funciona para filmes?**

_(sua resposta aqui)_

**2. Qual foi a limitação mais visível do baseline ortogonal (Parte 2) comparado ao Word2Vec?**

_(sua resposta aqui, usando o valor de similaridade que você calculou na Parte 2)_

**3. As recomendações fizeram sentido para o filme semente escolhido?**

_(sua resposta aqui: cite pelo menos um acerto e uma possível recomendação estranha, com uma hipótese do
porquê ela apareceu)_


## Checklist antes de entregar

Confira se o seu notebook:

- [ ] Executa do início ao fim sem erros, em uma sessão nova do Colab (`Ambiente de execução > Executar
  tudo`).
- [ ] Usa um filme semente diferente dos colegas nas Partes 4, 5 e 6.
- [ ] Contém a justificativa dos hiperparâmetros do Word2Vec (Parte 3), não apenas os valores.
- [ ] Contém o relatório da Parte 6 preenchido, com as três respostas.
- [ ] Foi salvo/compartilhado como link do Colab com permissão de visualização para qualquer pessoa com
  o link.

O gabarito está [disponível aqui](https://gist.github.com/rodrigoclira/82cc333012b3f303bd692346ba479812) para você comparar sua lógica e suas justificativas. Não se preocupe se os números exatos de similaridade forem diferentes dos do gabarito: isso é esperado devido à aleatoriedade do treinamento do Word2Vec.


## Referências

- Aula "Tokens e Embeddings", Tópicos Especiais em Inteligência Artificial, Prof. Rodrigo Lira, IFPE
  Campus Paulista.
- Harper, F. M.; Konstan, J. A. (2015). *The MovieLens Datasets: History and Context*. ACM Transactions
  on Interactive Intelligent Systems (TiiS). Dataset disponível em:
  https://grouplens.org/datasets/movielens/latest/
- Alammar, J.; Grootendorst, M. *Hands-On Large Language Models*, Capítulo 2. O'Reilly, 2024.
- Alammar, J. *The Illustrated Word2vec*. https://jalammar.github.io/illustrated-word2vec/
- Documentação do gensim Word2Vec: https://radimrehurek.com/gensim/models/word2vec.html
- Documentação do scikit-learn TfidfVectorizer:
  https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
